# PeerBERT: Colab runner

Runs the full pipeline on the two original datasets with the original models from the Hugging Face Hub.

1. **Runtime -> Change runtime type -> T4 GPU** (or better).
2. Run the cells from top to bottom.
3. After a disconnect, run cells 1-4 again, then continue where you stopped.

Every step is restart-safe: finished work is skipped and training resumes from the last epoch.
Outputs go to Google Drive (`MyDrive/PeerBERT_outputs`), so nothing is lost when the session ends.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Storage: Google Drive on Colab, `/kaggle/working` on Kaggle

In [ ]:
import os
ON_KAGGLE = os.path.exists("/kaggle/working")
if ON_KAGGLE:
    WORK, CODE = "/kaggle/working", "/kaggle/working/PeerBERT"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK, CODE = "/content/drive/MyDrive/PeerBERT_outputs", "/content/PeerBERT"
os.makedirs(WORK, exist_ok=True)
os.environ["PRAD_WORK_DIR"] = WORK
print("outputs ->", WORK)

## 3. Get or update the code

In [ ]:
REPO = "https://github.com/Wajahatali56/PeerBERT.git"   # private repo: https://<TOKEN>@github.com/...
BRANCH = "claude/brave-dirac-qt3mdw"
import subprocess
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 4. Install dependencies

Optional: add a Hugging Face token as a Colab secret named `HF_TOKEN` (key icon on the left).
Downloads are faster and not rate-limited.

In [ ]:
!pip install -q -r requirements.txt
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception:
    print("no HF_TOKEN secret (anonymous downloads)")
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())

## 5. (Optional) code self-test: about 3 minutes on CPU, no data downloads

In [ ]:
!pip install -q pytest
!python -m pytest -q tests

## 6. Load both datasets, split by paper, leakage audit

Expect `verify: PASS` for both datasets. If it reports `SchemaError` or `audit FAILED`, stop and read the message.
If Colab runs out of RAM, run `--dataset semdetect` and `--dataset intellabs` in separate cells.

In [ ]:
!python scripts/00_inspect_datasets.py --include-semdetect
!python scripts/02_prepare_splits.py
!python scripts/02_prepare_splits.py --verify-only

## 7. PeerBERT: 1-epoch smoke run first, then the 3-seed training

In [ ]:
!python scripts/03_train_intellabs.py --smoke

In [ ]:
!python scripts/03_train_intellabs.py

In [ ]:
!python scripts/04_train_semdetect.py

## 8. Evaluate PeerBERT

In [ ]:
!python scripts/05_evaluate.py --dataset intellabs
!python scripts/05_evaluate.py --dataset semdetect

## 9. Baselines (one group per cell keeps each run short)

Each group frees GPU memory before the next one, and finished baselines are skipped on re-run.

In [ ]:
!python scripts/06_run_baselines.py --dataset intellabs --only logrank gltr detectllm_lrr mfd fast_detectgpt
!python scripts/06_run_baselines.py --dataset semdetect --only logrank gltr detectllm_lrr mfd fast_detectgpt

In [ ]:
!python scripts/06_run_baselines.py --dataset intellabs --only radar binoculars
!python scripts/06_run_baselines.py --dataset semdetect --only radar binoculars

In [ ]:
!python scripts/06_run_baselines.py --dataset intellabs --only dnagpt
!python scripts/06_run_baselines.py --dataset semdetect --only dnagpt

## 10. Ablation (validation-selected, includes the CoCoDet-style margin loss)

In [ ]:
!python scripts/07_run_ablation.py --dataset intellabs
!python scripts/07_run_ablation.py --dataset semdetect

## 11. Cross-dataset transfer and explainability

In [ ]:
!python scripts/08_cross_dataset.py --source intellabs
!python scripts/08_cross_dataset.py --source semdetect
!python scripts/10_explain.py --dataset intellabs
!python scripts/10_explain.py --dataset semdetect

## 12. Final report

In [ ]:
!python scripts/09_build_report.py
from IPython.display import Markdown, display
display(Markdown(open(f"{WORK}/outputs/results/report/report.md").read()))